In [ ]:

import model as md
import numpy as np
import matplotlib.pyplot as plt
import importlib
import seaborn as sns
importlib.reload(md)


In [ ]:
#varry: rho, alpha, k1, k1-k2
M=15 #nb of simulation for averaging (precision)
n=500 #size of signal
k1= 5 
k2= 4
alpha = np.sqrt(0.5)

RHO = np.linspace(0,1,2)

M1 =  []
M2 = []
M2bis = []
for rho in RHO:
    over1 = []
    over2 = []
    over2bis = []
    for _ in range(M):
        D, x1, x2 = md.spec(n, k1, k2, rho, alpha) 
        m1, m2 , m2bis, _= md.overlap(D,x1,x2)
        over1.append(m1)
        over2.append(m2)
        over2bis.append(m2bis)
    M1.append([np.mean(over1), np.std(over1)/np.sqrt(M)])
    M2.append([np.mean(over2),np.std(over2)/np.sqrt(M)])
    M2bis.append([np.mean(over2bis),np.std(over2bis)/np.sqrt(M)])

M1 = np.array(M1)
M2 = np.array(M2)
M2bis = np.array(M2bis)

sns.set_theme(style="whitegrid", context="paper")
plt.errorbar(RHO,M1[:,0], yerr= M1[:,1], label = "x1.v1", capsize=2, linewidth = 1)
plt.errorbar(RHO,M2[:,0],yerr = M2[:,1], label = "x2.v2", capsize=2, linewidth = 1)
plt.errorbar(RHO,M2bis[:,0], yerr= M2bis[:,1], label = "x2.v1", capsize=2, linewidth = 1)
plt.ylabel("Overlaps", fontsize=16)
plt.xlabel(r"Correlation: $\rho$", fontsize=16)
plt.grid(alpha=0.5)
plt.legend(frameon=True, fontsize=16)
plt.tight_layout()
plt.tick_params(axis='both', labelsize=14)
plt.show()


In [ ]:
def run_experiment_3D(vary_param_x, vary_values_x, vary_param_y, vary_values_y, 
                       M, n, k1, k2, rho, alpha):
    """
    Parameters:
    -----------
    vary_param_x  : string, parametre sur l'axe x ('rho', 'k1', 'k2', 'alpha', 'n')
    vary_values_x : array, valeurs du parametre x
    vary_param_y  : string, parametre sur l'axe y
    vary_values_y : array, valeurs du parametre y
    M             : int, nombre de simulations
    n, k1, k2, rho, alpha : parametres fixes
    """
    
    Z1 = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z2 = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z2bis = np.zeros((len(vary_values_y), len(vary_values_x)))

    for i, val_y in enumerate(vary_values_y):
        for j, val_x in enumerate(vary_values_x):
            params = {'n': n, 'k1': k1, 'k2': k2, 'rho': rho, 'alpha': alpha}
            params[vary_param_x] = val_x
            params[vary_param_y] = val_y

            over1, over2, over2bis = [], [], []
            for _ in range(M):
                D, x1, x2 = md.spec(params['n'], params['k1'], params['k2'], 
                                      params['rho'], params['alpha'])
                m1, m2, m2bis = md.overlap(D, x1, x2)
                over1.append(m1)
                over2.append(m2)
                over2bis.append(m2bis)

            Z1[i, j] = np.mean(over1)
            Z2[i, j] = np.mean(over2)
            Z2bis[i, j] = np.mean(over2bis)

    # meshgrid pour le plot
    X, Y = np.meshgrid(vary_values_x, vary_values_y)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={'projection': '3d'})
    titles = ['x1.v1', 'x2.v2', 'x2.v1']
    Zs = [Z1, Z2, Z2bis]

    for ax, Z, title in zip(axes, Zs, titles):
        surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)
        ax.set_xlabel(vary_param_x, fontsize=12)
        ax.set_ylabel(vary_param_y, fontsize=12)
        ax.set_zlabel('Overlap', fontsize=12)
        ax.set_title(title, fontsize=14)
        fig.colorbar(surf, ax=ax, shrink=0.5)

    plt.tight_layout()
    plt.show()

    return Z1, Z2, Z2bis

In [ ]:
def run_experiment_2D(vary_param_x, vary_values_x, vary_param_y, vary_values_y, 
                       M, n, k1, k2, rho, alpha, all_overlap = True, plot = True):
    
    Z11    = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z22   = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z12 = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z21 = np.zeros((len(vary_values_y), len(vary_values_x)))

    for i, val_y in enumerate(vary_values_y):
        for j, val_x in enumerate(vary_values_x):
            params = {'n': n, 'k1': k1, 'k2': k2, 'rho': rho, 'alpha': alpha}
            params[vary_param_x] = val_x
            params[vary_param_y] = val_y

            over11, over22, over12, over21 = [], [], [], []
            for _ in range(M):
                D, x1, x2 = md.spec(params['n'], params['k1'], params['k2'], 
                                      params['rho'], params['alpha'])
                m11, m22, m12, m21 = md.overlap(D, x1, x2, all_overlap)
                over11.append(m11)
                over22.append(m22)
                over12.append(m12)
                over21.append(m21)

            Z11[i, j]    = np.mean(over11)
            Z21[i, j] = np.mean(over21)
            if all_overlap:
                Z22[i, j]    = np.mean(over22)
                Z12[i, j] = np.mean(over12)
                
    if plot : 
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()
        titles = ['x1.v1', 'x2.v2', 'x1.v2', 'x2.v1']
        Zs = [Z11, Z22, Z12, Z21]

        for ax, Z, title in zip(axes, Zs, titles):
            im = ax.imshow(Z, origin='lower', aspect='auto', cmap='viridis',
                        extent=[vary_values_x[0], vary_values_x[-1], 
                                vary_values_y[0], vary_values_y[-1]],vmin=0, vmax=1)
            fig.colorbar(im, ax=ax, label='Overlap')
            ax.set_xlabel(vary_param_x, fontsize=12)
            ax.set_ylabel(vary_param_y, fontsize=12)
            ax.set_title(title, fontsize=14)

        plt.tight_layout()
        plt.show()

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes = axes.flatten()
        titles = ['max over x1', 'max over x2']
        Zs = [np.maximum(Z11,Z12), np.maximum(Z22,Z21)]

        for ax, Z, title in zip(axes, Zs, titles):
            im = ax.imshow(Z, origin='lower', aspect='auto', cmap='viridis',
                        extent=[vary_values_x[0], vary_values_x[-1], 
                                vary_values_y[0], vary_values_y[-1]],vmin=0, vmax=1)
            fig.colorbar(im, ax=ax, label='Overlap')
            ax.set_xlabel(vary_param_x, fontsize=12)
            ax.set_ylabel(vary_param_y, fontsize=12)
            ax.set_title(title, fontsize=14)
        # plot f(k1) = sqrt( (k2/k1)^2 / (1 + (k2/k1)^2) )
        #    k1_vals = vary_values_x[vary_values_x > 1]
        #    k2=2
            
        #   f_vals = np.sqrt((k2 / k1_vals)**2 / (1 + (k2 / k1_vals)**2))
        #    ax.plot(k1_vals, f_vals, color='red', linewidth=2, label=r'$f(k_1)$')
        #    ax.legend(fontsize=12)
        plt.tight_layout()
        plt.show()
    

    return Z11, Z22, Z12, Z21

In [14]:
#varry: rho, alpha, k1, k1-k2
M=10 #nb of simulation for averaging (precision)
n=100 #size of signal
k1= 5
k2= 2
alpha = np.sqrt(0.5) #Y1 plus imp que Y2
rho = 0.2
RHO = np.linspace(0,1,30)
K1 = np.linspace(0,5,30)
K2 = np.linspace(0,5,30)
ALPHA = np.linspace(0,1,30)

Z11, Z22, Z12, Z21 = run_experiment_2D(
    vary_param_x='rho', vary_values_x=RHO,
    vary_param_y='k1',  vary_values_y=K1,
    M=M, n=n, k1=k1, k2=k2, rho=rho, alpha=alpha,all_overlap= False, plot = False
)



In [ ]:
im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[ALPHA[0], ALPHA[-1], 
                                RHO[0], RHO[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)

cbar.set_label(r" Overlap $m_1 = |\langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$ ", fontsize=14) 

plt.xlabel(r"$\alpha$", fontsize=14)
plt.ylabel(r"Correlation $\rho$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

im = plt.imshow(Z21, origin='lower', aspect='auto', cmap='viridis',
                        extent=[ALPHA[0], ALPHA[-1], 
                                RHO[0], RHO[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_2 = | \langle \mathbf{\hat{x}} , \mathbf{x}_2 \rangle | $", fontsize=14)

plt.xlabel(r"$\alpha$", fontsize=14)
plt.ylabel(r"Correlation $\rho$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                RHO[0], RHO[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_1 = | \langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$", fontsize=14) 

plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"Correlation $\rho$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

im = plt.imshow(Z21, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                RHO[0], RHO[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_2 = | \langle \mathbf{\hat{x}} , \mathbf{x}_2 \rangle |$", fontsize=14)

plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"Correlation $\rho$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:

im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                ALPHA[0], ALPHA[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_1 = | \langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$", fontsize=14) 

k1_vals = K1[K1 != 0]
k2= 2

k1_vals = K1.copy()
f_vals = np.ones_like(k1_vals)

mask = k1_vals != 0
f_vals[mask] = np.abs(k2 / k1_vals[mask]) / np.sqrt(1 + (k2 / k1_vals[mask])**2)


#plt.plot(k1_vals, f_vals, color='red', linewidth=2, label=r'$\alpha_* = f(\lambda_1)$')
#plt.legend(fontsize=14)
plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"$\alpha$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

im = plt.imshow(Z21, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                ALPHA[0], ALPHA[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_2 = | \langle \mathbf{\hat{x}} , \mathbf{x}_2 \rangle |$", fontsize=14)

#plt.plot(k1_vals, f_vals, color='red', linewidth=2, label=r'$\alpha_* = f(\lambda_1)$') 
#plt.legend(fontsize=14)
plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"$\alpha$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                K2[0], K2[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_1 = | \langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$", fontsize=14) 

plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"$\lambda_2$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

im = plt.imshow(Z21, origin='lower', aspect='auto', cmap='viridis',
                        extent=[K1[0], K1[-1], 
                                K2[0], K2[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_2 = | \langle \mathbf{\hat{x}} , \mathbf{x}_2 \rangle |$", fontsize=14)


plt.xlabel(r"$\lambda_1$", fontsize=14)
plt.ylabel(r"$\lambda_2$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[RHO[0], RHO[-1], 
                                K1[0], K1[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_1 = | \langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$", fontsize=14) 


lam = 1.01
lam1 = (np.sqrt(1- alpha**2)*k2*lam - lam**2)/((RHO**2 +1)*alpha*np.sqrt(1- alpha**2)*k2 - alpha*lam)

plt.plot(RHO, lam1, color='red', linewidth=2, label=r'$\alpha_* = f(\lambda_1)$') 
plt.legend(fontsize=14)
plt.xlabel(r"$\rho$", fontsize=14)
plt.ylabel(r"$\lambda_1$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

im = plt.imshow(Z21, origin='lower', aspect='auto', cmap='viridis',
                        extent=[RHO[0], RHO[-1], 
                                K1[0], K1[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_2 = | \langle \mathbf{\hat{x}} , \mathbf{x}_2 \rangle |$", fontsize=14)


plt.xlabel(r"$\rho$", fontsize=14)
plt.ylabel(r"$\lambda_1$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

def theta_plus(lambda1, lambda2, alpha, rho):
    a = alpha**2 * lambda1**2
    b = (1 - alpha**2) * lambda2**2
    cross = rho**2 * alpha**2 * (1 - alpha**2) * lambda1**2 * lambda2**2
    return (a + b) / 2 + np.sqrt(((a - b) / 2)**2 + cross)

def find_lambda1_critical(rho, lambda2, alpha, target=2.0):
    # theta_+(lambda1, ...) = target  =>  solve for lambda1
    f = lambda l1: theta_plus(l1, lambda2, alpha, rho) - target
    # theta_+ is increasing in lambda1, so bracket [0, large]
    return brentq(f, 1e-6, 100.0)

# --- Parameters ---
alpha   = 1 / np.sqrt(2)
lambda2 = 2.0
target  = 1

# --- Sweep rho ---
rhos     = np.linspace(0, 0.99, 500)
lambda1s = [find_lambda1_critical(r, lambda2, alpha, target) for r in rhos]


im = plt.imshow(Z11, origin='lower', aspect='auto', cmap='viridis',
                        extent=[RHO[0], RHO[-1], 
                                K1[0], K1[-1]],vmin=0, vmax=1)
cbar = plt.colorbar(im)
cbar.set_label(r" Overlap $m_1 = | \langle \mathbf{\hat{x}} , \mathbf{x}_1 \rangle |$", fontsize=14) 


plt.plot(rhos, lambda1s,'-r')

plt.legend(fontsize=14)
plt.xlabel(r"$\rho$", fontsize=14)
plt.ylabel(r"$\lambda_1$", fontsize=14)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.title(r"BBP critical curve: $\theta_+ = 2$"
          f"\n" + r"$\alpha={alpha:.2f},\ \lambda_2={lambda2}$")

plt.tight_layout()
plt.show()

ValueError: f(a) and f(b) must have different signs